# Grounded Generation Benchmark

Menjalankan benchmark dengan retrieval final dan `Qwen/Qwen2.5-3B-Instruct`. Lima kasus pertama menjadi gate terakhir untuk protokol baris: jika ada output generation invalid, runner berhenti dan format tuning base model tidak dilanjutkan. Jika gate lolos, runner meneruskan 60 kasus. Output otomatis mencakup retrieval, citation precision, valid-output rate, abstention, latency, dan VRAM. Faithfulness serta answer relevance tetap memerlukan audit manual atas predictions.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
BRANCH = "dev/apiip"

%cd /content
!test -d indonesian-legal-compliance-rag || git clone --depth 1 --branch {BRANCH} https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git
!git -C indonesian-legal-compliance-rag pull --ff-only origin {BRANCH}
%cd /content/indonesian-legal-compliance-rag
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y -q torchvision torchcodec
!python scripts/check_environment.py
!python -m unittest discover -s tests -v
!python -m eval.validate_cases --require-reviewed

In [ ]:
from pathlib import Path

corpus = [
    "PP Nomor 5 Tahun 2021.pdf",
    "PP Nomor 35 Tahun 2021.pdf",
    "PP Nomor 51 Tahun 2023.pdf",
    "UU Nomor 6 Tahun 2023.pdf",
]
if not all((Path("data/raw") / filename).is_file() for filename in corpus):
    !python -m gdown --folder https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql -O data/raw
assert all((Path("data/raw") / filename).is_file() for filename in corpus), "Corpus tidak lengkap"

In [ ]:
!python -m eval.run_grounded

In [ ]:
import json
from pathlib import Path

report_path = Path("eval/results/grounded-generation/grounded_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
{
    "case_count": report["case_count"],
    "format_gate_passed": report["format_gate_passed"],
    "metrics": report["metrics"],
}

In [ ]:
from google.colab import files

files.download("eval/results/grounded-generation/grounded_report.json")
files.download("eval/results/grounded-generation/grounded_predictions.jsonl")